In [54]:
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
import numpy as np
import requests
import time
import os
!pip install python-dotenv
from dotenv import load_dotenv

load_dotenv()


[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


True

In [2]:
onemap_api_key = os.environ.get('ONEMAP_API_KEY')

if not onemap_api_key:
    raise ValueError(
        "ONEMAP_API_KEY not found. Please create a .env file in this directory with: \n"
        "ONEMAP_API_KEY=your_api_key_here"
    )

In [9]:
# Get file names 
hdb_points_file = "../cleaned_datasets/resale_2019_to_2025_cleaned.geojson"
busstop_file = "LTABusStop.geojson"
output_file = "hdb_busstop_distances.csv"

### 1. Get walking distance between closest mall and HDB 

1.1 Run a sample of postal codes to ensure code works, check the result output to see if there are any nulls 

In [10]:
# --- 1. load and prep data ---
raw_hdb_gdf = gpd.read_file(hdb_points_file).to_crs(epsg=4326)
# take only the first 5 unique postals for this sample
postal_gdf_sample = raw_hdb_gdf.drop_duplicates(subset=['postal_code']).head(5).copy()

busstop_gdf = gpd.read_file(busstop_file).to_crs(epsg=4326)

busstop_coords = list(zip(busstop_gdf.geometry.y, busstop_gdf.geometry.x))
busstop_nums = busstop_gdf['BUS_STOP_NUM'].tolist() 

tree = cKDTree(busstop_coords)

results = []

print(f"--- starting sample test for {len(postal_gdf_sample)} postals ---")

# --- 2. sample loop ---
for idx, row in postal_gdf_sample.iterrows():
    p_code = str(row['postal_code'])
    origin_coords = (row.geometry.y, row.geometry.x)
    
    print(f"\nchecking postal: {p_code} at {origin_coords}")
    
    # get 3 closest bus stop by straight line
    dist, indices = tree.query(origin_coords, k=3) 
    
    best_walk = float('inf')
    closest_busstop = None

    for i in indices:
        dest_coords = busstop_coords[i]
        busstop_num = busstop_nums[i]
        
        
        # using the routingsvc endpoint from the docs
        url = f"https://www.onemap.gov.sg/api/public/routingsvc/route?start={origin_coords[0]},{origin_coords[1]}&end={dest_coords[0]},{dest_coords[1]}&routeType=walk"
        
        try:

            token = onemap_api_key if onemap_api_key.startswith("Bearer ") else f"Bearer {onemap_api_key}"
            headers = {"Authorization": token}
        
            res = requests.get(url, headers=headers, timeout=10)
            
            if res.status_code == 200:
                data = res.json()
                if 'route_summary' in data:
                    walk_m = data['route_summary']['total_distance']
                    print(f"  -> found route to {busstop_num}: {walk_m}m")
                    if walk_m < best_walk:
                        best_walk = walk_m
                        closest_busstop = busstop_num
                else:
                    print(f"  -> no route summary found for {busstop_num}")
            else:
                print(f"  -> api error {res.status_code}: {res.text}")
            
            time.sleep(0.2) # slightly longer sleep for the sample
        except Exception as e:
            print(f"  -> request failed: {e}")

    results.append({
        "postal_code": p_code,
        "nearest_busstop": closest_busstop,
        "walking_dist_m": best_walk if best_walk != float('inf') else None
    })

# --- 3. display output ---
sample_df = pd.DataFrame(results)
print("\n--- final sample output ---")
print(sample_df)

# check if we actually got results
if sample_df['walking_dist_m'].isnull().all():
    print("\nwarning: all walking distances are null. check your api key and coordinate order.")
else:
    print("\nsuccess: distances found! you can now run the full script.")

--- starting sample test for 5 postals ---

checking postal: 560225 at (1.3673961277036402, 103.83815000747873)
  -> found route to 54621: 453m
  -> found route to 54629: 571m
  -> found route to 54219: 442m

checking postal: 560174 at (1.3751965588189876, 103.83769584780252)
  -> found route to 54199: 445m
  -> found route to 54191: 257m
  -> found route to 54201: 415m

checking postal: 560440 at (1.3664278983686127, 103.85431149122876)
  -> found route to 54381: 245m
  -> found route to 54389: 385m
  -> found route to 54329: 568m

checking postal: 560637 at (1.3803292495669066, 103.8423366937204)
  -> found route to 55199: 210m
  -> found route to 55169: 249m
  -> found route to 55181: 370m

checking postal: 560571 at (1.3701655201821288, 103.85494058284989)
  -> found route to 54279: 228m
  -> found route to 54281: 260m
  -> found route to 54271: 194m

--- final sample output ---
  postal_code nearest_busstop  walking_dist_m
0      560225           54219             442
1      56017

1.2 Run full script if results generated for the sample postal codes

In [20]:
postal_gdf["postal_code"].nunique()

9650

In [19]:
# --- 1. Load Datasets (full) ---
raw_hdb_gdf = gpd.read_file(hdb_points_file).to_crs(epsg=4326)
postal_gdf = raw_hdb_gdf.drop_duplicates(subset=['postal_code']).copy()
postal_gdf['postal_code'] = postal_gdf['postal_code'].astype(str).str.zfill(6)

busstop_gdf = gpd.read_file(busstop_file).to_crs(epsg=4326)
busstop_coords = list(zip(busstop_gdf.geometry.y, busstop_gdf.geometry.x))
busstop_nums = busstop_gdf['BUS_STOP_NUM'].tolist()

tree = cKDTree(busstop_coords)

# --- 2. Resume logic ---
if os.path.exists(output_file):
    processed_df = pd.read_csv(output_file, dtype={'postal_code': str})
    processed_df['postal_code'] = processed_df['postal_code'].str.zfill(6)
    processed_postals = set(processed_df['postal_code'])
    results = processed_df.to_dict('records')
    print(f"Resuming... {len(processed_postals)} postals already completed.")
else:
    results = []
    processed_postals = set()

# --- 3. Full loop ---
print(f"Starting processing for {len(postal_gdf) - len(processed_postals)} remaining postals...")

try:
    for idx, row in postal_gdf.iterrows():
        p_code = row['postal_code']
        if p_code in processed_postals:
            continue

        origin_coords = (row.geometry.y, row.geometry.x)
        dist, indices = tree.query(origin_coords, k=3)
        
        best_walk = float('inf')
        best_busstop = None

        for i in indices:
            dest_coords = busstop_coords[i]
            token = onemap_api_key if onemap_api_key.startswith("Bearer ") else f"Bearer {onemap_api_key}"
            url = f"https://www.onemap.gov.sg/api/public/routingsvc/route?start={origin_coords[0]},{origin_coords[1]}&end={dest_coords[0]},{dest_coords[1]}&routeType=walk"
            
            try:
                res = requests.get(url, headers={"Authorization": token}, timeout=10)
                if res.status_code == 200:
                    data = res.json()
                    if 'route_summary' in data:
                        walk_m = data['route_summary']['total_distance']
                        if walk_m < best_walk:
                            best_walk = walk_m
                            best_busstop = busstop_nums[i]
                time.sleep(0.2)
            except:
                continue

        results.append({
            "postal_code": p_code,
            "nearest_busstop": best_busstop,
            "walking_dist_m": best_walk if best_walk != float('inf') else None
        })
        processed_postals.add(p_code)

        if len(results) % 50 == 0:
            pd.DataFrame(results).to_csv(output_file, index=False)
            print(f"progress: {len(results)} / {len(postal_gdf)} unique postals saved.")
            

except KeyboardInterrupt:
    print("\nPaused. Saving current progress...")

# Final save
pd.DataFrame(results).to_csv(output_file, index=False)
print(f"Task complete! Results saved to {output_file}")

Resuming... 9650 postals already completed.
Starting processing for 0 remaining postals...
Task complete! Results saved to hdb_busstop_distances.csv


In [ ]:
hdb_busstop_distances = pd.read_csv("../cleaned_datasets/hdb_busstop_distances.csv")
hdb_busstop_distances.info()

<class 'pandas.DataFrame'>
RangeIndex: 9650 entries, 0 to 9649
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   postal_code      9650 non-null   int64  
 1   nearest_busstop  9650 non-null   int64  
 2   walking_dist_m   9650 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 226.3 KB


### 3. Get feature variables of busstops

- num_busstops_500m
- num_busstops_1km

In [ ]:
# read files
unique_hdb = gpd.read_file(hdb_points_file).copy()
busstop_gdf = gpd.read_file(busstop_file).copy()
# hdb_busstop_distances = pd.read_csv("../cleaned_datasets/hdb_busstop_distances.csv")
hdb_busstop_distances = pd.read_csv("../datasets/hdb_busstop_distances.csv")

# get unique hdb locations
unique_hdb = unique_hdb.drop_duplicates(subset=["postal_code"]).copy()
unique_hdb = unique_hdb.to_crs(epsg=3414)
busstop_gdf = busstop_gdf.to_crs(epsg=3414)

# prepare unique hdb and bus stop coordinates for distance calculation
hdb_coords = np.array([(geom.x, geom.y) for geom in unique_hdb.geometry])
busstop_coords = np.array([(geom.x, geom.y) for geom in busstop_gdf.geometry])

busstop_nums = busstop_gdf["BUS_STOP_NUM"].astype(str).tolist()

# build cKDTree for efficient spatial queries
tree = cKDTree(busstop_coords)

# get bus stops within 500m and 1km
indices_500m = tree.query_ball_point(hdb_coords, r=500)
indices_1km = tree.query_ball_point(hdb_coords, r=1000)

# convert index arrays into bus stop number lists
unique_hdb["busstops_within_500m"] = [
    [busstop_nums[i] for i in idx_list] for idx_list in indices_500m]

unique_hdb["busstops_within_1km"] = [
    [busstop_nums[i] for i in idx_list] for idx_list in indices_1km]

# count columns
unique_hdb["num_busstops_500m"] = unique_hdb["busstops_within_500m"].apply(len)
unique_hdb["num_busstops_1km"] = unique_hdb["busstops_within_1km"].apply(len)

busstop_counts = unique_hdb[[
    "postal_code",
    "num_busstops_500m",
    "num_busstops_1km"
]].copy()

print(busstop_counts)

       postal_code  num_busstops_500m  num_busstops_1km
0           560225                 14                48
1           560174                 12                50
2           560440                 13                51
4           560637                 19                61
5           560571                 18                57
...            ...                ...               ...
178362      763478                  9                30
178365      761478                  9                30
179299      761795                 13                56
179458      762479                 10                37
179459      763479                 10                37

[9650 rows x 3 columns]


### 4. Merge with nearest bus stop walking-distance table and save file

In [72]:
# convert postal_code to string for merging
hdb_busstop_distances["postal_code"] = hdb_busstop_distances["postal_code"].astype(str).str.zfill(6)
busstop_counts["postal_code"] = busstop_counts["postal_code"].astype(str).str.zfill(6)
hdb_busstop_distances["nearest_busstop"] = hdb_busstop_distances["nearest_busstop"].astype(str)

hdb_busstop_final = hdb_busstop_distances.merge(
    busstop_counts,
    on="postal_code",
    how="left"
)

print(hdb_busstop_final.head())
print(hdb_busstop_final.info())

  postal_code nearest_busstop  walking_dist_m  num_busstops_500m  \
0      560406           54319           158.0                 13   
1      560314           54331           292.0                 11   
2      561308           54019           130.0                 13   
3      560323           54247           154.0                 15   
4      560174           54191           257.0                 12   

   num_busstops_1km  
0                37  
1                55  
2                50  
3                53  
4                50  
<class 'pandas.DataFrame'>
RangeIndex: 9650 entries, 0 to 9649
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   postal_code        9650 non-null   str    
 1   nearest_busstop    9650 non-null   str    
 2   walking_dist_m     9650 non-null   float64
 3   num_busstops_500m  9650 non-null   int64  
 4   num_busstops_1km   9650 non-null   int64  
dtypes: float64(1), int64(2)

In [73]:
hdb_busstop_final.to_csv("../cleaned_datasets/hdb_busstop_distances.csv", index=False)